In [ ]:
import os
import matplotlib.pyplot as plt
import keras
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

## 1. Load data (grayscale, resize)

In [ ]:
UKURAN = 250
data_path = "Rice"
kelas = sorted(os.listdir(data_path))
print(kelas)

x, y = [], []
for label, nama_kelas in enumerate(kelas):
    folder = os.path.join(data_path, nama_kelas)
    print(folder)
    for nama_file in os.listdir(folder):
        if nama_file.endswith("Thumbs.db"):
            continue
        gambar = keras.utils.load_img(os.path.join(folder, nama_file), color_mode="grayscale", target_size=(UKURAN, UKURAN))
        x.append(np.array(gambar))
        y.append(label)

x = np.array(x)
y = np.array(y)

In [ ]:
print("Jumlah gambar:", len(x))
print("Ukuran x:", x.shape)
print("Ukuran y:", y.shape)

## 2. Preprocessing (normalisasi + one-hot)

In [ ]:
x = x.astype("float32") / 255.0
y = keras.utils.to_categorical(y, num_classes=len(kelas))

print(x.shape)
print(y.shape)
print(y[0])

## 3. Split 80% train / 10% validation / 10% test

In [ ]:
# 80% train, 20% temporary
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42)

# 20% temporary -> 10% validation, 10% test
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

print("Train:", len(x_train))
print("Validation:", len(x_val))
print("Test:", len(x_test))

## 4. Model DCNN
Conv -> Pool -> Conv -> Conv -> Pool -> Flatten -> Dense (Dropout hanya setelah Dense)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(UKURAN, UKURAN, 1)),

    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),

    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),

    keras.layers.Flatten(),

    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(kelas), activation="softmax"),
])

model.summary()

## 5. Training
Sesuaikan `batch_size` dan `epochs` dengan soal (default aslab: 100 dan 20).

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=100, epochs=20, validation_data=(x_val, y_val))

## 6. Predict (9 gambar test)

In [ ]:
prediksi = model.predict(x_test)
tebakan = np.argmax(prediksi, axis=1)
real = np.argmax(y_test, axis=1)

plt.figure(figsize=(15, 15))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    plt.title(f"Tebakan: {kelas[tebakan[i]]} \n Asli: {kelas[real[i]]}")
    plt.axis("off")

plt.show()

## 7. Evaluate

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print("Loss di data test:", loss)
print("Akurasi di data test:", accuracy)

## 8. Plot accuracy dan loss

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.title("Loss")
plt.legend()

plt.show()